# Notebook 17 — WINDOW-E2-001: SCV-matched five-record feature ablation

**Research decision:** The SCV-IDS paper's own window-size ablation reports **75.82% accuracy at window size 1 and 85.77% at window size 5** in its paper evaluation. Our strongest previous tabular models instead used only the *last* record of a five-row window. This ablation tests the **original information difference directly**, rather than guessing another architecture.

## Exactly matched, leakage-aware comparison

**Frozen source:** UNSW-NB15 **original training CSV only**, SHA256 checked. **Frozen development train/validation split:** 140,332 / 35,009 CSV rows, producing exactly **140,328 / 35,005 labeled sliding windows**, with label of the fifth row and all 42 features at each of five split-local positions. Feature shape: **210 columns**, of which 15 are native categorical variables, with training-only vocabularies. All labels, record IDs, and official test columns excluded. The official 82,332-row test is unopened.

Two **new** candidate fits: unweighted and rare-class-balanced LightGBM over all 210 window features. Compare with the **previously frozen** last-record LightGBM versions on exactly the same 35,005 validation window targets. The paper's 85.77% result comes from an entirely different evaluation partition; any internal-validation score comparison is **not** paper superiority.

**Critical semantic limitation:** Source CSV order does not establish real time or authentic flow continuity. These are *serialized row-order windows*, NOT verified temporal trajectories. A successful E2 would require a later shuffled-context/chronological control before claiming temporal mechanisms.

**Crash recovery:** Real tqdm boosting bars; every 100 rounds save a temporary local partial booster. Every *completed* model is independently audited against its true predictions, backed up in an unpublished private GitHub draft Release, and aggregate metrics committed to public GitHub **before starting the next model**. Previously archived candidates are restored on a fresh Kaggle version run with SHA256 validation. If Kaggle crashes **during** a model before remote archive, partial local checkpoint recovery is not guaranteed.

## Kaggle setup

Attach `dsshayan/unsw-nb15`, Internet ON, enable the existing Kaggle secret `pushRare`, Accelerator **None (CPU)**. Use **Save Version → Save & Run All** so the run is independent of your open browser. One code cell. Final output banner: `WINDOW-E2-001 FINAL TRAINING AND GITHUB BACKUPS COMPLETE`.

In [ ]:
# ONE CELL: original frozen split -> native 5x42 tabular feature model -> immediate backup
import os,sys,subprocess,hashlib
from pathlib import Path
from kaggle_secrets import UserSecretsClient

REPO=Path("/kaggle/working/rare-attack-ids-window-e2")
BASE=Path("/kaggle/working/research_outputs")
RUN=BASE/"WINDOW-E2-001-S42"
TRAIN_CSV=Path("/kaggle/input/datasets/dsshayan/unsw-nb15/UNSW_NB15_training-set.csv")
assert TRAIN_CSV.is_file(),"Attach dsshayan/unsw-nb15 (training-set.csv)"
with TRAIN_CSV.open("rb") as file:
    assert hashlib.file_digest(file,"sha256").hexdigest()==(
        "bec7dd5ec88dc2a0ccc7a07879d338395ed7421750f675fd0339e07dfe0648fa"
    ),"Input training CSV content differs from frozen research source"
secret=UserSecretsClient().get_secret("pushRare")
assert secret,"Authorize the existing Kaggle secret pushRare for this notebook"
if not REPO.is_dir():
    subprocess.run(["git","clone","--depth","1",
        "https://github.com/itsCodeBakery/rare-attack-spatiotemporal-ids.git",
        str(REPO)],check=True)
else:
    subprocess.run(["git","-C",str(REPO),"pull","--ff-only"],check=True)
env=os.environ.copy()
env["GITHUB_TOKEN"]=secret
env["PYTHONPATH"]=str(REPO/"src")
env["OMP_NUM_THREADS"]="4"
env["OPENBLAS_NUM_THREADS"]="1"
print("E2 EXPERIMENT SOURCE COMMIT:",
    subprocess.check_output(["git","-C",str(REPO),"rev-parse","HEAD"],text=True).strip(),flush=True)

for script,archive in (
    ("restore_scv_release.py","SCV-IDS-REPRO-001_FULL_ARTIFACTS.tar"),
    ("restore_ctrl003_tabular.py","CTRL-TAB-001-S42_FULL_ARTIFACTS.tar")
):
    subprocess.run([
        sys.executable,str(REPO/"scripts"/script),"--base",str(BASE),
        "--archive",str(Path("/kaggle/working")/archive)
    ],cwd=str(REPO),env=env,check=True)

subprocess.run([
    sys.executable,str(REPO/"scripts/run_window_e2.py"),
    "--repo",str(REPO),
    "--reference",str(BASE/"SCV-IDS-REPRO-001"),
    "--baseline",str(BASE/"CTRL-TAB-001-S42"),
    "--config",str(REPO/"configs/window_e2_001_seed42.json"),
    "--output",str(RUN),
    "--auto-archive"
],cwd=str(REPO),env=env,check=True)
print("\\nWINDOW-E2-001 VERIFIED FINAL COMPARISON:")
print((RUN/"comparison_all_models.csv").read_text(),flush=True)
print("All candidate models privately backed up in draft releases.",flush=True)
